# MCP Filesystem Client (stdio transport)

This notebook demonstrates how to connect to the official `@modelcontextprotocol/server-filesystem` Node.js server as a subprocess and call its tools from pure Python.

**What is the Model Context Protocol (MCP)?**
MCP is an open standard that enables developers to build secure, two-way connections between AI models and local or remote resources (like filesystems, databases, or enterprise tools). It standardizes how tools are discovered and invoked.

In this notebook, we will use the **stdio transport** layer. This means our Python client will start the Node.js server as a background subprocess and communicate with it using standard input and standard output streams via JSON-RPC messages.

We are using a Jupyter Notebook to break down the client-server interaction step-by-step. To maintain the asynchronous connection across multiple notebook cells, we will utilize Python's `contextlib.AsyncExitStack`.

In [5]:
import asyncio
import os
from contextlib import AsyncExitStack
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import subprocess

# ── Configuration ─────────────────────────────────────────────────────────────
ALLOWED_DIR = os.path.expanduser("~/mcp_sandbox")
os.makedirs(ALLOWED_DIR, exist_ok=True)

print(f"Sandbox directory ready at: {ALLOWED_DIR}")

# Server launch parameters - we use npx so no global npm install is needed.
# It will execute the filesystem server and only allow access to ALLOWED_DIR.
SERVER_PARAMS = StdioServerParameters(
    command="npx",
    args=["-y", "@modelcontextprotocol/server-filesystem", ALLOWED_DIR],
)
print("Server parameters configured.")

Sandbox directory ready at: C:\Users\ashutoshb/mcp_sandbox
Server parameters configured.


### Helper Function for Tool Calls

The `ClientSession.call_tool` method returns a result object containing a list of content blocks (which could be text, images, or binary blobs).
To make our interactions easier to read, we define a thin helper function `call` that takes care of invoking a tool and extracting just the textual content from the server's response. This helps declutter our demonstration code later on.

In [6]:
async def call(session: ClientSession, tool: str, **kwargs):
    """Thin wrapper: call a tool and return the first text result."""
    print(f"[Client] Calling tool '{tool}' with args: {kwargs}")
    
    # Invoking the tool on the MCP server
    result = await session.call_tool(tool, arguments=kwargs)
    
    # Extracting the text from the result blocks
    texts = [block.text for block in result.content if hasattr(block, "text")]
    return "\n".join(texts)

### Connecting to the Server and Handshake

Here is where the actual connection happens. In a standard Python script, you would use `async with` context managers to cleanly start and stop the server subprocess.
However, because we want to inspect the connection step-by-step in different notebook cells, we use an `AsyncExitStack`. This allows us to "enter" the context manager and keep the server running in the background while we execute subsequent cells.

1. We launch the `stdio_client` with our server parameters. This starts the Node process and binds to its stdio streams.
2. We wrap those communication streams in an MCP `ClientSession`.
3. We perform the **initialization handshake** (`session.initialize()`). This is a crucial step in the MCP protocol where the client and server exchange capabilities, register supported features, and agree on protocol versions.

In [7]:
stack = AsyncExitStack()

print("Starting stdio client...")
read, write = await stack.enter_async_context(stdio_client(SERVER_PARAMS, errlog=subprocess.DEVNULL))

print("Creating client session...")
session = await stack.enter_async_context(ClientSession(read, write))

print("Initializing session (Handshake)...")
await session.initialize()
print("Connected and initialized successfully!")

Starting stdio client...
Creating client session...
Initializing session (Handshake)...
Connected and initialized successfully!


### Discovering Available Tools

One of the key features of MCP is discoverability. The client does not need to know the tools beforehand; it can ask the server, "What tools do you support?"

We use the `list_tools()` method to retrieve a list of all available filesystem operations provided by the server. Each tool comes with a name, a description, and a JSON Schema defining its required arguments (which we don't print here for brevity, but the AI uses to know what parameters to send).

In [ ]:
tools_response = await session.list_tools()
print("Available tools on the server:")
for tool in tools_response.tools:
    print(f"   * {tool.name}: {tool.description}")

### Using the Tools: Writing a File

Now that we know what tools are available, let's invoke one. We will use the `write_file` tool to create a new text file inside our sandboxed directory.

Notice how we pass the tool name and its expected arguments (path and content) to our helper function. The MCP server will execute the filesystem operation on our behalf, ensuring it stays within the allowed directory limits we defined at startup.

In [ ]:
file_path = os.path.join(ALLOWED_DIR, "hello.txt")
print(f"Writing file to path: {file_path}")

result = await call(
    session,
    "write_file",
    path=file_path,
    content="Hello from Python MCP client!\nThis is an automated test file.\n",
)
print(f"Result from server: {result}")

### Using the Tools: Reading the File Back

To verify our write operation was successful, we use the `read_file` tool.
The server will read the file from the local disk and return its contents securely over the MCP stdio connection.

In [ ]:
print(f"Reading file from path: {file_path}")
content = await call(session, "read_file", path=file_path)
print(f"File Content:\n{content}")

### Using the Tools: Listing Directory Contents

The `list_directory` tool allows us to inspect the contents of our sandbox folder. This is analogous to running `ls` or `dir` in a terminal. It returns the files and folders currently present in the specified directory.

In [ ]:
print(f"Listing directory: {ALLOWED_DIR}")
listing = await call(session, "list_directory", path=ALLOWED_DIR)
print(f"Directory Listing:\n{listing}")

### Using the Tools: Creating a Directory and Moving Files

MCP servers can handle a variety of operations sequentially. Here, we'll demonstrate two tools:
1. `create_directory`: Creates a new subdirectory named 'subdir'.
2. `move_file`: Renames or moves our existing 'hello.txt' file into the newly created subdirectory.

In [ ]:
sub_dir = os.path.join(ALLOWED_DIR, "subdir")
print(f"Creating directory: {sub_dir}")
result = await call(session, "create_directory", path=sub_dir)
print(f"Create dir result: {result}\n")

new_path = os.path.join(sub_dir, "hello_moved.txt")
print(f"Moving {file_path} to {new_path}")
result = await call(
    session, "move_file", source=file_path, destination=new_path
)
print(f"Move file result: {result}")

### Using the Tools: Searching for Files

The filesystem server also provides a robust way to search for files using glob patterns. The `search_files` tool scans the directory and returns a list of files that match our `*.txt` pattern.

In [ ]:
print(f"Searching for '*.txt' under {ALLOWED_DIR}")
found = await call(
    session, "search_files", path=ALLOWED_DIR, pattern="*.txt"
)
print(f"Search results:\n{found}")

### Using the Tools: Inspecting File Metadata

Finally, we can retrieve detailed metadata about a file (like its size, creation time, and modification time) using the `get_file_info` tool. This allows an AI to understand more context about the files it is working with.

In [ ]:
print(f"Getting file info for: {new_path}")
info = await call(session, "get_file_info", path=new_path)
print(f"Info Details: {info}")

### Cleanup: Closing the Session

Once we are done interacting with the server, it is important to close the session gracefully. This shuts down the Node.js subprocess and frees up system resources. We do this by asynchronously closing our `AsyncExitStack`.

In [ ]:
print("Closing the session and shutting down the server...")
await stack.aclose()
print("Cleanup complete! Server subprocess terminated.")